# Steppable-Surface Segmentation of a Simulated LiDAR Staircase
### Self-contained notebook — runs in Google Colab with no uploads

This notebook generates a synthetic 4-step staircase point cloud, segments it
into **steppable treads / risers / other** with four classical geometric methods,
evaluates them, and studies robustness to noise.

**How to run:** `Runtime ▸ Run all` (or run top-to-bottom). All code is defined
inline in Section 1 — there is **no `src/` package to upload**. Only NumPy,
scikit-learn and Matplotlib are needed, all of which are pre-installed in Colab.

## 1. Project code — run this section once
These cells define everything (config, data generator, the four methods, evaluation, plotting). Run them top-to-bottom before Section 2.

### 1.1 Configuration & label scheme

In [ ]:
from __future__ import annotations
"""Configuration objects for the staircase point-cloud segmentation pipeline.

All geometry is expressed in metres. The label scheme is shared across the
data generator, the segmentation methods and the evaluation utilities so that
predictions and ground truth are always comparable.
"""

from dataclasses import dataclass, field

# --- Semantic label scheme (single source of truth) -------------------------
STEPPABLE: int = 0   # horizontal tread surface a foot/wheel can land on
RISER: int = 1       # vertical face between two treads
OTHER: int = 2       # edges, outliers and unstructured LiDAR noise

LABEL_NAMES: dict[int, str] = {STEPPABLE: "steppable", RISER: "riser", OTHER: "other"}
LABEL_COLORS: dict[int, tuple[float, float, float]] = {
    STEPPABLE: (0.15, 0.70, 0.25),  # green
    RISER: (0.85, 0.18, 0.18),      # red
    OTHER: (0.55, 0.55, 0.55),      # grey
}


@dataclass(frozen=True)
class StaircaseConfig:
    """Parametric description of a 4-step staircase and its LiDAR noise model."""

    n_steps: int = 4
    width: float = 2.5            # x-axis extent (2.0 - 3.0 m)
    depth: float = 0.30           # tread depth per step (0.25 - 0.35 m)
    height: float = 0.18          # rise per step (0.15 - 0.20 m)

    # Point budget (total target >= 15_000 per the brief).
    points_per_tread: int = 3000
    points_per_riser: int = 1800

    # Noise model -----------------------------------------------------------
    tread_z_sigma: float = 0.02   # vertical jitter on treads (from the brief)
    riser_y_sigma: float = 0.02   # depth jitter on risers
    lidar_sigma: float = 0.006    # isotropic ranging noise applied to every point
    outlier_ratio: float = 0.03   # fraction of total points injected as outliers
    edge_ratio: float = 0.02      # fraction added as structured edge clutter

    seed: int = 42

    @property
    def total_rise(self) -> float:
        return self.n_steps * self.height

    @property
    def total_run(self) -> float:
        return self.n_steps * self.depth


@dataclass
class SegmentationParams:
    """Hyper-parameters shared by the geometric segmentation methods."""

    # Per-point / per-cluster normal classification thresholds (degrees from +z).
    horizontal_max_angle: float = 30.0   # |angle(n, z)| <= this  -> horizontal
    vertical_min_angle: float = 60.0     # |angle(n, z)| >= this  -> vertical

    normal_k: int = 50                   # k-NN used for local normal estimation

    # RANSAC plane fitting. Band ~2*sigma_z captures a full tread in one plane;
    # the high min-inlier count rejects thin horizontal slices through risers.
    # RANSAC uses its OWN tight orientation tolerance so it never accepts a
    # plane near the oblique staircase incline (~31 deg from horizontal).
    ransac_threshold: float = 0.04       # inlier distance (m)
    ransac_iters: int = 600
    ransac_max_planes: int = 12
    ransac_min_inliers: int = 1000
    ransac_horizontal_max_angle: float = 15.0
    ransac_vertical_min_angle: float = 75.0

    # DBSCAN (normal-augmented feature space: [x, y, z, w*nx, w*ny, w*nz]).
    # Uses a LARGER normal neighbourhood than the per-point method: smoother
    # normals make perpendicular surfaces separate cleanly in feature space,
    # while a high weight w keeps treads and risers in distinct clusters.
    dbscan_eps: float = 0.20
    dbscan_min_samples: int = 20
    dbscan_normal_weight: float = 2.0    # scales normals into the clustering space
    dbscan_normal_k: int = 80            # k-NN for DBSCAN's (smoothed) normals

    # Height-histogram analysis.
    hist_bin: float = 0.01               # z-histogram resolution (m)
    hist_band: float = 0.04              # half-width of a tread band around a peak

    rng_seed: int = 0

### 1.2 Data generation

In [ ]:
from __future__ import annotations
"""Synthetic staircase point-cloud generation.

Implements the mathematical model from the brief. For each step ``i``:

Tread (horizontal, steppable)::

    x ~ U(0, width)
    y ~ U(i * depth, (i + 1) * depth)
    z = i * height + eps_z,        eps_z ~ N(0, tread_z_sigma^2)

Riser (vertical face joining tread ``i-1`` to tread ``i``)::

    x ~ U(0, width)
    y = i * depth + eps_y,         eps_y ~ N(0, riser_y_sigma^2)
    z ~ U((i - 1) * height, i * height)

On top of the structural model we add isotropic LiDAR ranging noise, a band of
structured edge clutter and a population of uniform outliers, each carrying the
correct ground-truth semantic label.
"""

import numpy as np



def _tread_points(i: int, cfg: StaircaseConfig, rng: np.random.Generator) -> np.ndarray:
    n = cfg.points_per_tread
    x = rng.uniform(0.0, cfg.width, n)
    y = rng.uniform(i * cfg.depth, (i + 1) * cfg.depth, n)
    z = i * cfg.height + rng.normal(0.0, cfg.tread_z_sigma, n)
    return np.column_stack((x, y, z))


def _riser_points(i: int, cfg: StaircaseConfig, rng: np.random.Generator) -> np.ndarray:
    n = cfg.points_per_riser
    x = rng.uniform(0.0, cfg.width, n)
    y = i * cfg.depth + rng.normal(0.0, cfg.riser_y_sigma, n)
    z = rng.uniform((i - 1) * cfg.height, i * cfg.height, n)
    return np.column_stack((x, y, z))


def _edge_points(cfg: StaircaseConfig, n: int, rng: np.random.Generator) -> np.ndarray:
    """Structured clutter along the lateral edges of the staircase (x ~ 0 or width)."""
    side = rng.integers(0, 2, n)
    x = np.where(side == 0, 0.0, cfg.width) + rng.normal(0.0, 0.01, n)
    y = rng.uniform(0.0, cfg.total_run, n)
    z = rng.uniform(0.0, cfg.total_rise, n) + rng.normal(0.0, 0.03, n)
    return np.column_stack((x, y, z))


def _outlier_points(cfg: StaircaseConfig, n: int, rng: np.random.Generator) -> np.ndarray:
    """Uniform outliers spread through an inflated bounding box (LiDAR ghost returns)."""
    x = rng.uniform(-0.2, cfg.width + 0.2, n)
    y = rng.uniform(-0.2, cfg.total_run + 0.2, n)
    z = rng.uniform(-0.1, cfg.total_rise + 0.25, n)
    return np.column_stack((x, y, z))


def generate_staircase(
    cfg: StaircaseConfig | None = None,
    lidar_sigma: float | None = None,
    outlier_ratio: float | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Generate a labelled staircase point cloud.

    Parameters
    ----------
    cfg:
        Geometry / noise configuration. Defaults to :class:`StaircaseConfig`.
    lidar_sigma, outlier_ratio:
        Optional overrides, convenient for noise-sensitivity sweeps without
        rebuilding a whole config object.

    Returns
    -------
    points : (N, 3) float64 array
    labels : (N,) int array with values in {STEPPABLE, RISER, OTHER}
    """
    cfg = cfg or StaircaseConfig()
    lidar_sigma = cfg.lidar_sigma if lidar_sigma is None else lidar_sigma
    outlier_ratio = cfg.outlier_ratio if outlier_ratio is None else outlier_ratio

    rng = np.random.default_rng(cfg.seed)

    clouds: list[np.ndarray] = []
    labels: list[np.ndarray] = []

    # Treads: i = 0 .. n_steps - 1
    for i in range(cfg.n_steps):
        pts = _tread_points(i, cfg, rng)
        clouds.append(pts)
        labels.append(np.full(len(pts), STEPPABLE, dtype=np.int64))

    # Risers: i = 1 .. n_steps - 1 (vertical faces connecting consecutive treads)
    for i in range(1, cfg.n_steps):
        pts = _riser_points(i, cfg, rng)
        clouds.append(pts)
        labels.append(np.full(len(pts), RISER, dtype=np.int64))

    structural_n = sum(len(c) for c in clouds)

    n_edge = int(cfg.edge_ratio * structural_n)
    if n_edge:
        pts = _edge_points(cfg, n_edge, rng)
        clouds.append(pts)
        labels.append(np.full(len(pts), OTHER, dtype=np.int64))

    n_out = int(outlier_ratio * structural_n)
    if n_out:
        pts = _outlier_points(cfg, n_out, rng)
        clouds.append(pts)
        labels.append(np.full(len(pts), OTHER, dtype=np.int64))

    points = np.vstack(clouds)
    labels_arr = np.concatenate(labels)

    # Isotropic LiDAR ranging noise on every point.
    if lidar_sigma > 0:
        points = points + rng.normal(0.0, lidar_sigma, points.shape)

    # Shuffle so downstream methods cannot exploit point ordering.
    perm = rng.permutation(len(points))
    return points[perm], labels_arr[perm]

### 1.3 Segmentation methods

In [ ]:
from __future__ import annotations
"""Geometric segmentation of staircase point clouds.

Four self-contained methods are provided, each mapping a point cloud to a
per-point label in {STEPPABLE, RISER, OTHER}:

1. ``segment_by_normals``   - per-point local-PCA normals + orientation rules.
2. ``segment_ransac``       - sequential multi-plane RANSAC + plane orientation.
3. ``segment_dbscan_slope`` - DBSCAN clustering + per-cluster slope analysis.
4. ``segment_height_histogram`` - z-histogram tread-level detection + normals.

All methods depend only on NumPy and scikit-learn so the notebook runs in any
environment; Open3D is used purely for optional interactive visualisation.
"""

import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors



# --------------------------------------------------------------------------- #
# Shared geometry helpers                                                     #
# --------------------------------------------------------------------------- #
def estimate_normals(points: np.ndarray, k: int = 30) -> np.ndarray:
    """Estimate unit normals via PCA over each point's k-nearest neighbourhood.

    The normal is the eigenvector of the local covariance matrix associated
    with the smallest eigenvalue. Normals are sign-disambiguated towards +z so
    that ``|n_z|`` is directly comparable across points.
    """
    n = len(points)
    k = min(k, n)
    nn = NearestNeighbors(n_neighbors=k).fit(points)
    _, idx = nn.kneighbors(points)

    neighbours = points[idx]                                   # (n, k, 3)
    centered = neighbours - neighbours.mean(axis=1, keepdims=True)
    cov = np.einsum("nki,nkj->nij", centered, centered) / k    # (n, 3, 3)

    eigvals, eigvecs = np.linalg.eigh(cov)                     # ascending eigvals
    normals = eigvecs[:, :, 0]                                 # smallest-eigval vector

    # Disambiguate sign so the z-component is non-negative.
    flip = normals[:, 2] < 0
    normals[flip] *= -1.0
    return normals


def _classify_by_normal_z(nz_abs: np.ndarray, params: SegmentationParams) -> np.ndarray:
    """Map |cos(angle between normal and z)| to a semantic label."""
    cos_h = np.cos(np.deg2rad(params.horizontal_max_angle))
    cos_v = np.cos(np.deg2rad(params.vertical_min_angle))
    labels = np.full(len(nz_abs), OTHER, dtype=np.int64)
    labels[nz_abs >= cos_h] = STEPPABLE
    labels[nz_abs <= cos_v] = RISER
    return labels


def _plane_from_points(p: np.ndarray) -> tuple[np.ndarray, float]:
    """Least-squares plane (unit normal, offset d) through >=3 points: n·x + d = 0.

    Uses the eigendecomposition of the 3x3 covariance (O(n) memory) rather than
    a full SVD, whose left-singular matrix would be (n, n) for large clusters.
    """
    centroid = p.mean(axis=0)
    centered = p - centroid
    cov = centered.T @ centered
    eigvals, eigvecs = np.linalg.eigh(cov)
    normal = eigvecs[:, 0]               # smallest-eigenvalue direction
    d = -normal @ centroid
    return normal, d


# --------------------------------------------------------------------------- #
# Method 1 - per-point normal / PCA analysis                                  #
# --------------------------------------------------------------------------- #
def segment_by_normals(
    points: np.ndarray, params: SegmentationParams | None = None
) -> np.ndarray:
    params = params or SegmentationParams()
    normals = estimate_normals(points, params.normal_k)
    return _classify_by_normal_z(np.abs(normals[:, 2]), params)


# --------------------------------------------------------------------------- #
# Method 2 - sequential multi-plane RANSAC                                     #
# --------------------------------------------------------------------------- #
def _ransac_single_plane(
    points: np.ndarray,
    params: SegmentationParams,
    rng: np.random.Generator,
    horizontal: bool,
    cos_h: float,
    cos_v: float,
) -> np.ndarray:
    """Return a boolean inlier mask for the dominant plane of the requested
    orientation. Candidate planes of the wrong orientation are rejected during
    scoring, so RANSAC never locks onto the oblique staircase-incline plane.
    """
    n = len(points)
    best_inliers = np.zeros(n, dtype=bool)
    best_count = 0
    for _ in range(params.ransac_iters):
        sample = points[rng.choice(n, 3, replace=False)]
        normal, d = _plane_from_points(sample)
        if not np.isfinite(normal).all():
            continue
        nz = abs(normal[2])
        ok = (nz >= cos_h) if horizontal else (nz <= cos_v)
        if not ok:
            continue
        dist = np.abs(points @ normal + d)
        inliers = dist < params.ransac_threshold
        count = int(inliers.sum())
        if count > best_count:
            best_count, best_inliers = count, inliers
    return best_inliers


def _extract_planes(
    points: np.ndarray,
    remaining: np.ndarray,
    labels: np.ndarray,
    target_label: int,
    horizontal: bool,
    params: SegmentationParams,
    rng: np.random.Generator,
) -> None:
    """Greedily extract axis-aligned planes of one orientation, in place.

    ``horizontal=True`` accepts only near-horizontal candidates (treads);
    ``horizontal=False`` accepts only near-vertical candidates (risers).
    """
    cos_h = np.cos(np.deg2rad(params.ransac_horizontal_max_angle))
    cos_v = np.cos(np.deg2rad(params.ransac_vertical_min_angle))
    for _ in range(params.ransac_max_planes):
        idx = np.flatnonzero(remaining)
        if len(idx) < params.ransac_min_inliers:
            break
        mask = _ransac_single_plane(points[idx], params, rng, horizontal, cos_h, cos_v)
        if int(mask.sum()) < params.ransac_min_inliers:
            break
        plane_idx = idx[mask]
        labels[plane_idx] = target_label
        remaining[plane_idx] = False


def segment_ransac(
    points: np.ndarray, params: SegmentationParams | None = None
) -> np.ndarray:
    """Two-phase multi-plane RANSAC.

    Risers (vertical planes) are extracted first and removed, because each
    riser's top edge is coplanar in z with the tread above it; extracting
    treads first would otherwise sweep those riser points into the steppable
    class. Remaining structure is then segmented into horizontal tread planes.
    """
    params = params or SegmentationParams()
    rng = np.random.default_rng(params.rng_seed)

    labels = np.full(len(points), OTHER, dtype=np.int64)
    remaining = np.ones(len(points), dtype=bool)

    _extract_planes(points, remaining, labels, RISER, horizontal=False, params=params, rng=rng)
    _extract_planes(points, remaining, labels, STEPPABLE, horizontal=True, params=params, rng=rng)
    return labels


# --------------------------------------------------------------------------- #
# Method 3 - DBSCAN clustering + per-cluster slope analysis                    #
# --------------------------------------------------------------------------- #
def segment_dbscan_slope(
    points: np.ndarray, params: SegmentationParams | None = None
) -> np.ndarray:
    """Normal-aware DBSCAN, then per-cluster slope classification.

    Raw-xyz DBSCAN merges the whole staircase into one connected component
    because adjacent treads and risers touch. We therefore cluster in the
    augmented space ``[x, y, z, w*n]`` so that two perpendicular surfaces that
    are spatially adjacent are still separated by their differing normals.
    """
    params = params or SegmentationParams()
    normals = estimate_normals(points, params.dbscan_normal_k)
    features = np.hstack([points, params.dbscan_normal_weight * normals])

    db = DBSCAN(
        eps=params.dbscan_eps,
        min_samples=params.dbscan_min_samples,
        algorithm="kd_tree",          # explicit: avoid the 'auto'->brute O(n^2) blow-up
        n_jobs=1,
    )
    cluster_ids = db.fit_predict(features)

    labels = np.full(len(points), OTHER, dtype=np.int64)
    cos_h = np.cos(np.deg2rad(params.horizontal_max_angle))
    cos_v = np.cos(np.deg2rad(params.vertical_min_angle))

    for cid in np.unique(cluster_ids):
        if cid == -1:                       # DBSCAN noise stays OTHER
            continue
        mask = cluster_ids == cid
        if mask.sum() < 3:
            continue
        normal, _ = _plane_from_points(points[mask])
        nz = abs(normal[2])
        if nz >= cos_h:
            labels[mask] = STEPPABLE
        elif nz <= cos_v:
            labels[mask] = RISER
        # else: leave as OTHER (oblique / mixed cluster)
    return labels


# --------------------------------------------------------------------------- #
# Method 4 - height-histogram tread detection (advanced)                       #
# --------------------------------------------------------------------------- #
def _detect_z_peaks(z: np.ndarray, params: SegmentationParams) -> np.ndarray:
    edges = np.arange(z.min(), z.max() + params.hist_bin, params.hist_bin)
    counts, edges = np.histogram(z, bins=edges)
    centers = 0.5 * (edges[:-1] + edges[1:])
    if len(counts) < 3:
        return centers[np.argmax(counts)][None] if len(counts) else np.array([])
    # Local maxima above an adaptive prominence threshold.
    thr = 0.25 * counts.max()
    is_peak = (
        (counts[1:-1] >= counts[:-2])
        & (counts[1:-1] >= counts[2:])
        & (counts[1:-1] >= thr)
    )
    peaks = centers[1:-1][is_peak]
    return peaks


def segment_height_histogram(
    points: np.ndarray, params: SegmentationParams | None = None
) -> np.ndarray:
    """Detect discrete tread levels from the z-histogram, then refine with normals."""
    params = params or SegmentationParams()
    z = points[:, 2]
    peaks = _detect_z_peaks(z, params)

    labels = np.full(len(points), OTHER, dtype=np.int64)
    if peaks.size:
        near_peak = np.min(np.abs(z[:, None] - peaks[None, :]), axis=1) <= params.hist_band
    else:
        near_peak = np.zeros(len(points), dtype=bool)

    # A point is steppable only if it sits in a tread band *and* is locally flat.
    normals = estimate_normals(points, params.normal_k)
    nz = np.abs(normals[:, 2])
    cos_h = np.cos(np.deg2rad(params.horizontal_max_angle))
    cos_v = np.cos(np.deg2rad(params.vertical_min_angle))

    labels[near_peak & (nz >= cos_h)] = STEPPABLE
    labels[(~near_peak) & (nz <= cos_v)] = RISER
    return labels


METHODS = {
    "normals": segment_by_normals,
    "ransac": segment_ransac,
    "dbscan_slope": segment_dbscan_slope,
    "height_histogram": segment_height_histogram,
}

### 1.4 Evaluation

In [ ]:
from __future__ import annotations
"""Performance evaluation for staircase segmentation.

Metrics follow the brief: overall accuracy plus precision, recall, F1 and IoU
for the safety-critical *steppable* class. A small noise-sensitivity sweep is
included to quantify robustness against LiDAR noise and outliers.
"""

from dataclasses import asdict, dataclass

import numpy as np



@dataclass
class Metrics:
    accuracy: float
    precision: float        # steppable class
    recall: float           # steppable class
    f1: float               # steppable class
    iou: float              # steppable class

    def as_row(self) -> dict[str, float]:
        return {k: round(v, 4) for k, v in asdict(self).items()}


def steppable_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Metrics:
    """Binary metrics for the steppable class, plus multi-class accuracy."""
    accuracy = float((y_true == y_pred).mean())

    gt_pos = y_true == STEPPABLE
    pr_pos = y_pred == STEPPABLE
    tp = float((gt_pos & pr_pos).sum())
    fp = float((~gt_pos & pr_pos).sum())
    fn = float((gt_pos & ~pr_pos).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) else 0.0
    return Metrics(accuracy, precision, recall, f1, iou)


def confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """3x3 confusion matrix ordered [STEPPABLE, RISER, OTHER]."""
    classes = [STEPPABLE, RISER, OTHER]
    cm = np.zeros((3, 3), dtype=np.int64)
    for i, ct in enumerate(classes):
        for j, cp in enumerate(classes):
            cm[i, j] = int(((y_true == ct) & (y_pred == cp)).sum())
    return cm


def evaluate_methods(
    points: np.ndarray, y_true: np.ndarray, methods: dict, params=None
) -> dict[str, Metrics]:
    out: dict[str, Metrics] = {}
    for name, fn in methods.items():
        y_pred = fn(points, params) if params is not None else fn(points)
        out[name] = steppable_metrics(y_true, y_pred)
    return out


def noise_sweep(
    method,
    sigmas: list[float],
    outlier_ratios: list[float],
    base_cfg: StaircaseConfig | None = None,
    params=None,
) -> dict[str, list[tuple[float, float]]]:
    """Sweep LiDAR sigma and outlier ratio independently; report (level, F1).

    Returns a dict with keys ``"lidar_sigma"`` and ``"outlier_ratio"`` mapping to
    lists of ``(level, steppable_f1)`` tuples.
    """
    base_cfg = base_cfg or StaircaseConfig()
    result: dict[str, list[tuple[float, float]]] = {"lidar_sigma": [], "outlier_ratio": []}

    for s in sigmas:
        pts, lab = generate_staircase(base_cfg, lidar_sigma=s)
        pred = method(pts, params) if params is not None else method(pts)
        result["lidar_sigma"].append((s, steppable_metrics(lab, pred).f1))

    for o in outlier_ratios:
        pts, lab = generate_staircase(base_cfg, outlier_ratio=o)
        pred = method(pts, params) if params is not None else method(pts)
        result["outlier_ratio"].append((o, steppable_metrics(lab, pred).f1))

    return result

### 1.5 Visualization

In [ ]:
from __future__ import annotations
"""Visualisation helpers (matplotlib for headless PNG export, Open3D optional)."""

import numpy as np



def _colors_for(labels: np.ndarray) -> np.ndarray:
    c = np.zeros((len(labels), 3))
    for k, col in LABEL_COLORS.items():
        c[labels == k] = col
    return c


def plot_pointcloud_mpl(
    points: np.ndarray,
    labels: np.ndarray,
    title: str = "",
    ax=None,
    elev: float = 22.0,
    azim: float = -60.0,
    s: float = 1.5,
):
    """3D scatter coloured by label. Returns the matplotlib Axes."""
    import matplotlib.pyplot as plt  # local import keeps module import light

    if ax is None:
        fig = plt.figure(figsize=(7, 6))
        ax = fig.add_subplot(111, projection="3d")

    ax.scatter(
        points[:, 0], points[:, 1], points[:, 2],
        c=_colors_for(labels), s=s, depthshade=True, linewidths=0,
    )
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.set_zlabel("z (m)")
    ax.set_title(title)
    ax.view_init(elev=elev, azim=azim)
    try:
        ax.set_box_aspect(
            (np.ptp(points[:, 0]), np.ptp(points[:, 1]), np.ptp(points[:, 2]))
        )
    except Exception:
        pass
    return ax


def save_comparison_figure(
    points: np.ndarray,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    path: str,
    pred_title: str = "Prediction",
):
    """Side-by-side ground-truth vs prediction figure with a shared legend."""
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    fig = plt.figure(figsize=(13, 6))
    ax1 = fig.add_subplot(121, projection="3d")
    ax2 = fig.add_subplot(122, projection="3d")
    plot_pointcloud_mpl(points, y_true, "Ground truth", ax=ax1)
    plot_pointcloud_mpl(points, y_pred, pred_title, ax=ax2)

    handles = [
        Line2D([0], [0], marker="o", linestyle="", markerfacecolor=col,
               markeredgecolor=col, label=LABEL_NAMES[k])
        for k, col in LABEL_COLORS.items()
    ]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return path


def show_open3d(points: np.ndarray, labels: np.ndarray):  # pragma: no cover
    """Interactive Open3D viewer (optional dependency)."""
    try:
        import open3d as o3d
    except ImportError as exc:  # pragma: no cover
        raise ImportError("Install open3d for interactive visualisation: pip install open3d") from exc

    pc = o3d.geometry.PointCloud()
    pc.points = o3d.utility.Vector3dVector(points)
    pc.colors = o3d.utility.Vector3dVector(_colors_for(labels))
    o3d.visualization.draw_geometries([pc])

## 2. Generate the staircase point cloud

For step $i = 0..3$, with width $w$, depth $d$, height $h$:

**Tread (steppable):**
$$x \sim \mathcal{U}(0,w),\quad y \sim \mathcal{U}(i d,(i{+}1)d),\quad z = i h + \epsilon_z,\ \epsilon_z\sim\mathcal{N}(0,0.02^2)$$

**Riser (vertical):**
$$x \sim \mathcal{U}(0,w),\quad y = i d + \epsilon_y,\quad z \sim \mathcal{U}((i{-}1)h, i h)$$

Plus isotropic LiDAR noise, uniform outliers, and edge clutter.

In [ ]:
cfg = StaircaseConfig()
points, gt = generate_staircase(cfg)
print(f"{len(points)} points")
for lbl in (STEPPABLE, RISER, OTHER):
    print(f"  {LABEL_NAMES[lbl]:>9}: {(gt == lbl).sum()}")

### 2.1 Visualize the raw cloud (ground truth)

In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
plot_pointcloud_mpl(points, gt, title="Ground-truth labels", ax=ax)
plt.tight_layout(); plt.show()

## 3. Segmentation
Four geometric methods, each returning a per-point label in {steppable, riser, other}.

In [ ]:
params = SegmentationParams()
pred_normals = segment_by_normals(points, params)
m = steppable_metrics(gt, pred_normals)
print("PCA-normal method, steppable metrics:")
print(m)

### 3.1 Run and compare all four methods

In [ ]:
results = evaluate_methods(points, gt, METHODS, params)
print(f"{'method':<18}{'acc':>7}{'P':>8}{'R':>8}{'F1':>8}{'IoU':>8}")
for name, mt in results.items():
    print(f"{name:<18}{mt.accuracy:>7.3f}{mt.precision:>8.3f}{mt.recall:>8.3f}{mt.f1:>8.3f}{mt.iou:>8.3f}")
best = max(results, key=lambda k: results[k].f1)
print("\nBest by F1:", best)

### 3.2 Before / after visualization (best method)

In [ ]:
pred_best = METHODS[best](points, params)
fig = plt.figure(figsize=(13, 6))
ax1 = fig.add_subplot(121, projection="3d")
plot_pointcloud_mpl(points, gt, title="Before (ground truth)", ax=ax1)
ax2 = fig.add_subplot(122, projection="3d")
plot_pointcloud_mpl(points, pred_best, title=f"After ({best})", ax=ax2)
plt.tight_layout(); plt.show()

### 3.3 Confusion matrix (best method)

In [ ]:
cm = confusion_matrix(gt, pred_best)
print("rows = ground truth, cols = prediction, order [steppable, riser, other]")
print(cm)

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels([LABEL_NAMES[i] for i in range(3)], rotation=45, ha="right")
ax.set_yticklabels([LABEL_NAMES[i] for i in range(3)])
ax.set_xlabel("predicted"); ax.set_ylabel("ground truth")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

## 4. Noise and outlier sensitivity

We sweep the LiDAR noise $\sigma_L$ and the outlier ratio $\rho$, re-generating the
cloud each time, and record steppable F1 for the best method.

In [ ]:
sigmas = [0.0, 0.005, 0.01, 0.02, 0.03, 0.05]
outlier_ratios = [0.0, 0.03, 0.06, 0.10, 0.15, 0.20]
sweep = noise_sweep(METHODS[best], sigmas, outlier_ratios, base_cfg=cfg, params=params)
lid = np.array(sweep["lidar_sigma"]); out = np.array(sweep["outlier_ratio"])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(lid[:, 0], lid[:, 1], "o-"); axes[0].set_xlabel("LiDAR sigma (m)")
axes[0].set_ylabel("steppable F1"); axes[0].set_title("Effect of LiDAR noise"); axes[0].grid(alpha=.3)
axes[1].plot(out[:, 0], out[:, 1], "s-", color="C1"); axes[1].set_xlabel("outlier ratio")
axes[1].set_ylabel("steppable F1"); axes[1].set_title("Effect of outliers"); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print("LiDAR sweep (sigma, F1):", [(round(s, 3), round(f, 3)) for s, f in sweep["lidar_sigma"]])
print("Outlier sweep (rho, F1):", [(round(s, 3), round(f, 3)) for s, f in sweep["outlier_ratio"]])

## 5. Discussion: is this safe for a stair-following robot?

For foot placement the error cost is **asymmetric** — stepping where there is no
tread is dangerous, skipping a valid tread is not. The decisive metric is therefore
**steppable precision**, not raw accuracy.

- The PCA-normal method gives the best F1/IoU and recall, but is the most
  noise-sensitive because per-point normals inherit the full surface jitter.
- DBSCAN and the height-histogram reach the **highest steppable precision (~0.98)**:
  a robot acting on their output would almost never be told to step on a non-tread.
- Outliers are tolerated well; **LiDAR noise is the binding constraint**.

**Verdict:** at clean-to-moderate noise, a precision-oriented geometric method,
combined with a planner that only commits to high-confidence treads, is a
reasonable basis for a cautious stair-climbing controller.

## 6. (Optional) Interactive 3D with Open3D
Open3D is **not** installed in Colab by default and its interactive window does not
display inside Colab. This cell therefore just skips gracefully. To try it locally,
run `pip install open3d` on your own machine.

In [ ]:
try:
    show_open3d(points, pred_best)
except Exception as e:
    print("Open3D view skipped:", e)